# 03 · Training and evaluating the model

**Supports agenda block 6** ("Training & Evaluating the Model"). LightGBM
with walk-forward cross-validation - never a random shuffle-split, which
would train on the future and validate on the past - and the
**Information Coefficient (IC)**, HAC-corrected for the autocorrelation
that a 21-day-overlapping label mechanically introduces, as the metric
that actually says whether the model found anything.

In [1]:
import lightgbm as lgb
import pandas as pd
from ml4t.diagnostic.metrics import compute_ic_hac_stats, cross_sectional_ic_series
from ml4t.diagnostic.splitters import WalkForwardCV

DATA_DIR = "../data"
LABEL_HORIZON = 21  # trading days - must match the label built in 02_features_labels

dataset = pd.read_parquet(f"{DATA_DIR}/model_dataset.parquet")
feature_cols = [
    "mom_21d",
    "mom_63d",
    "mom_126d",
    "mom_252d",
    "vol_21d",
    "vol_63d",
    "rsi_14",
    "dollar_vol_rank",
]

dataset["timestamp"] = pd.to_datetime(dataset["timestamp"]).dt.tz_localize("UTC")
dataset = dataset.sort_values("timestamp").set_index("timestamp")
X, y = dataset[feature_cols], dataset["fwd_ret_21d"]

## Walk-forward cross-validation

`label_horizon=21` is the load-bearing argument here - it tells the
splitter that a training row's label is only *fully known* 21 trading
days after its feature date, so it purges the training rows whose label
window would otherwise overlap the validation period. Skip this argument
and the "CV" silently leaks the validation period's own future into
training, which is exactly the kind of mistake a coding agent will
reproduce faithfully if you don't specify it in the brief.

In [2]:
cv = WalkForwardCV(
    n_splits=16,
    test_size="1Y",
    label_horizon=LABEL_HORIZON,
    expanding=True,
    consecutive=True,
)

lgb_params = dict(
    objective="regression",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=200,
    verbosity=-1,
)

fold_predictions = []
for fold, (train_idx, test_idx) in enumerate(cv.split(X)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test = X.iloc[test_idx]

    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(X_train, y_train)

    preds = dataset.iloc[test_idx][["symbol", "fwd_ret_21d"]].copy()
    preds["prediction"] = model.predict(X_test)
    preds["fold"] = fold
    fold_predictions.append(preds)

    print(
        f"fold {fold}: train {X_train.index.min().date()}..{X_train.index.max().date()} "
        f"({len(X_train):,} rows) -> test {X_test.index.min().date()}..{X_test.index.max().date()} "
        f"({len(X_test):,} rows)"
    )

fold 0: train 2008-01-03..2008-12-01 (9,702 rows) -> test 2009-01-02..2009-12-31 (13,355 rows)


fold 1: train 2008-01-03..2009-12-01 (22,837 rows) -> test 2010-01-04..2010-12-31 (16,128 rows)


fold 2: train 2008-01-03..2010-12-01 (38,734 rows) -> test 2011-01-03..2011-12-30 (17,388 rows)


fold 3: train 2008-01-03..2011-11-30 (56,017 rows) -> test 2012-01-03..2013-01-03 (18,900 rows)


fold 4: train 2008-01-03..2012-12-03 (74,791 rows) -> test 2013-01-04..2014-01-03 (19,652 rows)


...

fold 11: train 2008-01-03..2019-12-05 (227,331 rows) -> test 2020-01-08..2021-01-06 (23,692 rows)


fold 12: train 2008-01-03..2020-12-04 (251,053 rows) -> test 2021-01-07..2022-01-05 (23,919 rows)


fold 13: train 2008-01-03..2021-12-06 (274,973 rows) -> test 2022-01-06..2023-01-06 (23,944 rows)


fold 14: train 2008-01-03..2022-12-06 (298,895 rows) -> test 2023-01-09..2024-01-09 (24,132 rows)


fold 15: train 2008-01-03..2023-12-07 (323,058 rows) -> test 2024-01-10..2025-01-10 (23,394 rows)


## The Information Coefficient

Every prediction above came from a fold where the model never saw that
period during training - this is out-of-sample by construction, not by
promise. We pool all four test folds and compute the cross-sectional
Spearman IC per date, then HAC-correct the resulting t-statistic.

In [3]:
oos = pd.concat(fold_predictions).reset_index().rename(columns={"index": "timestamp"})

ic_series = cross_sectional_ic_series(
    oos,
    oos,
    pred_col="prediction",
    ret_col="fwd_ret_21d",
    date_col="timestamp",
    entity_col="symbol",
    method="spearman",
)
print(ic_series.describe())

             n_obs           ic
count  4032.000000  4032.000000
mean     83.785962     0.008605
std      12.813146     0.250690
min      52.000000    -0.776163
25%      75.750000    -0.163195
50%      89.000000     0.017761
75%      95.000000     0.186282
max      96.000000     0.748541


In [4]:
hac_stats = compute_ic_hac_stats(ic_series, ic_col="ic", label_horizon=LABEL_HORIZON)
print(hac_stats)

{'mean_ic': 0.008605411609362887, 'hac_se': 0.013138290892592417, 't_stat': 0.6549871425220732, 'p_value': 0.5125134052096798, 'n_periods': 4032, 'effective_lags': 20, 'naive_se': 0.003947989560482749, 'naive_t_stat': 2.179694621155645}


`hac_stats["mean_ic"]` is the honest headline number - not the naive
t-statistic you'd get from treating each daily IC observation as
independent. Because the label is a 21-day forward return, consecutive
daily ICs share ~20 days of the same underlying return window and are
mechanically autocorrelated; the naive t-stat overstates significance.
`label_horizon=21` tells the HAC correction the minimum lag to account
for, rather than relying on order selection alone.

**Run this notebook and the two t-stats disagree with each other**: naive
≈ 2.18 (nominally "significant" at 5%), HAC ≈ 0.65 (nowhere close). That
gap *is* the lesson, not a bug to fix - eight simple technical features
on a monthly-rebalanced ETF panel do not reliably beat noise once the
autocorrelation induced by the overlapping 21-day label is priced in.
A model with an IC this weak has no business going anywhere near a
backtest that claims a live edge; block 7 puts it through one anyway, on
purpose, to show what "weak signal, meet real costs" actually looks like.

## Feature importance

Cheap to compute, easy to over-read. Treat this as "what the last fold's
model leaned on," not as a causal or stable ranking across the full
18-year history - `13_model_analysis.py` in the full case study goes much
further (SHAP, permutation importance, stability across folds) than this
workshop has time for.

In [5]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance)

mom_252d           499
vol_63d            437
mom_63d            390
dollar_vol_rank    371
vol_21d            368
mom_126d           363
mom_21d            193
rsi_14             179
dtype: int32


**Next:** `04_backtest.ipynb` - turn these predictions into positions and
run a cost-aware backtest. Save the pooled out-of-sample predictions so
the backtest notebook doesn't need to retrain anything.

In [6]:
oos.to_parquet(f"{DATA_DIR}/oos_predictions.parquet", index=False)